In [3]:
%pip install -q requests python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import requests
from dotenv import load_dotenv


# --------------------------------------------------
# LOAD OUR SECRET API KEYS
# --------------------------------------------------

# load_dotenv() reads the .env file
# and makes our API keys available to Python.
#
# Example .env:
#
# GEMINI_API_KEY=your_key
# GROQ_API_KEY=your_key
# OPENROUTER_API_KEY=your_key

load_dotenv()


# --------------------------------------------------
# CREATE OUR AI FUNCTION
# --------------------------------------------------

# "def" means DEFINE.
#
# We are creating our own function called "ask".
#
# Think of a function like a machine:
#
#     INPUT  →  MACHINE  →  OUTPUT
#
# provider = which AI should we use?
# prompt   = what question should we ask?
#
# Example:
#
# ask("gemini", "What is Python?")

def ask(provider, prompt):

    """
    Our AI waiter.

    provider = the AI we want to talk to
    prompt   = the question we want to ask

    Example:

        ask("gemini", "Explain APIs")
    """


    # --------------------------------------------------
    # 1. CHOOSE WHICH AI WE WANT TO USE
    # --------------------------------------------------

    # "if" means:
    #
    # "If this condition is true, do this."

    if provider == "gemini":

        # The Gemini API address
        url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent"

        # Our API key is like our ticket.
        #
        # os.getenv() means:
        # "Python, go and get this value from .env"

        headers = {
            "x-goog-api-key": os.getenv("GEMINI_API_KEY")
        }

        # This is the question we are sending to Gemini.
        #
        # "prompt" contains whatever question
        # the user gave us.

        body = {
            "contents": [
                {
                    "parts": [
                        {"text": prompt}
                    ]
                }
            ]
        }


    # --------------------------------------------------
    # GROQ
    # --------------------------------------------------

    # "elif" means:
    #
    # "ELSE IF"
    #
    # In simple English:
    #
    # "If the previous condition was NOT true,
    # check this one."

    elif provider == "groq":

        # Groq API address
        url = "https://api.groq.com/openai/v1/chat/completions"

        # Our Groq API key
        headers = {
            "Authorization": f"Bearer {os.getenv('GROQ_API_KEY')}",
            "Content-Type": "application/json"
        }

        # The question we are sending
        body = {
            "model": "openai/gpt-oss-20b",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }


    # --------------------------------------------------
    # OPENROUTER
    # --------------------------------------------------

    # If it was NOT Gemini
    # AND it was NOT Groq,
    # check if it is OpenRouter.

    elif provider == "openrouter":

        # OpenRouter API address
        url = "https://openrouter.ai/api/v1/chat/completions"

        # Our OpenRouter API key
        headers = {
            "Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}",
            "Content-Type": "application/json"
        }

        # openrouter/free means:
        #
        # "Give me an available free model."
        #
        # OpenRouter chooses an available free model
        # instead of us choosing one manually.

        body = {
            "model": "openrouter/free",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }


    # --------------------------------------------------
    # OLLAMA
    # --------------------------------------------------

    # If it was NOT Gemini,
    # NOT Groq,
    # NOT OpenRouter,
    # check if it is Ollama.

    elif provider == "ollama":

        # Ollama runs on our own computer.
        #
        # That's why we use localhost.
        #
        # localhost = "this computer"

        url = "http://localhost:11434/api/chat"

        # Ollama does not need an API key
        # because it is running locally.

        headers = {
            "Content-Type": "application/json"
        }

        # The question we are sending to Ollama

        body = {
            "model": "llama3.2",
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "stream": False
        }


    # --------------------------------------------------
    # UNKNOWN AI
    # --------------------------------------------------

    # "else" means:
    #
    # "None of the above worked."
    #
    # For example:
    #
    # ask("facebook", "Explain APIs")
    #
    # We don't have Facebook in our function,
    # so Python will come here.

    else:
        raise ValueError("Unknown provider")


    # --------------------------------------------------
    # 2. SEND THE QUESTION TO THE AI
    # --------------------------------------------------

    # requests.post() sends our question
    # to the AI server.
    #
    # Think of POST as:
    #
    # "Here is my question. Please process it."

    response = requests.post(
        url,
        headers=headers,
        json=body
    )


    # --------------------------------------------------
    # 3. CHECK FOR ERRORS
    # --------------------------------------------------

    # If the server says something went wrong,
    # this will show the error.

    response.raise_for_status()


    # --------------------------------------------------
    # 4. READ THE AI'S RESPONSE
    # --------------------------------------------------

    # The AI sends its answer back as JSON.
    #
    # .json() converts that response into
    # something Python can understand.

    data = response.json()


    # --------------------------------------------------
    # 5. GET THE ACTUAL ANSWER
    # --------------------------------------------------

    # Gemini puts its answer in a different
    # place inside the JSON response.

    if provider == "gemini":

        return data["candidates"][0]["content"]["parts"][0]["text"]


    # Ollama also has a different response format.

    elif provider == "ollama":

        return data["message"]["content"]


    # Groq and OpenRouter use this format.

    else:

        return data["choices"][0]["message"]["content"]

In [7]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

print("=" * 50)
print("STEP 1: Is the key loading at all?")
print("=" * 50)

if not api_key:
    print("❌ GEMINI_API_KEY is None or empty.")
    print("   -> Check that a file named exactly '.env' sits in the")
    print("      SAME folder you run this script from.")
    print("   -> Check the line inside .env reads exactly:")
    print("      GEMINI_API_KEY=your_key_here   (no quotes, no spaces around =)")
    exit()
else:
    print(f"✅ Key loaded. Starts with: {api_key[:6]}... (length {len(api_key)})")

print()
print("=" * 50)
print("STEP 2: What models does this key actually have access to?")
print("=" * 50)

list_url = "https://generativelanguage.googleapis.com/v1beta/models"
resp = requests.get(list_url, headers={"x-goog-api-key": api_key})

print(f"Status code: {resp.status_code}")

if resp.status_code != 200:
    print("❌ Could not list models. Response below:")
    print(resp.text)
else:
    data = resp.json()
    print("✅ Models available to this key that support generateContent:\n")
    for m in data.get("models", []):
        if "generateContent" in m.get("supportedGenerationMethods", []):
            print(" -", m["name"])

STEP 1: Is the key loading at all?
✅ Key loaded. Starts with: AQ.Ab8... (length 53)

STEP 2: What models does this key actually have access to?
Status code: 200
✅ Models available to this key that support generateContent:

 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemin

In [2]:
ask("gemini", "Explain APIs")

'An **API**, which stands for **Application Programming Interface**, is a set of rules and protocols that allows different software applications to communicate and share data with one another. \n\nIn simple terms, **an API is a messenger** that takes a request from one system, delivers it to another system, and then brings the response back.\n\n---\n\n### The Best Way to Understand: The Restaurant Analogy\n\nImagine you are sitting at a table in a restaurant. \n\n1. **You (The Client/User):** You want to order food. You are looking at the menu.\n2. **The Kitchen (The Server/Database):** This is where the food (the data) is prepared. But you can’t go directly into the kitchen to get it.\n3. **The Waiter (The API):** The waiter is the messenger. \n   * You give your order (Request) to the waiter.\n   * The waiter delivers the order to the kitchen.\n   * The kitchen prepares your food.\n   * The waiter brings the food (Response) back to you.\n\nWithout the waiter (API), you wouldn\'t be a

In [26]:
ask("groq", "Explain APIs")

'# What is an API?\n\n**API** stands for **Application Programming Interface**.  \nIt’s a set of rules, protocols, and tools that let one software program talk to another. Think of it as a *menu* in a restaurant:\n\n| Restaurant | API | Customer |\n|------------|-----|----------|\n| Kitchen (the underlying system that does the work) | Functions & endpoints (the “ingredients” and “recipes”) | Your code (the “order”) |\n| Server (the waiter) | Request/response format (e.g., HTTP, gRPC) | The menu you read (JSON, XML) |\n| Dish served | Data you get back | The final product you use in your app |\n\nIn other words, an API defines *how* to ask for a service or data, and *what* you’ll receive back, all in a standardized format that both sides understand.\n\n---\n\n## Core Concepts\n\n| Concept | What it means | Why it matters |\n|---------|---------------|----------------|\n| **Endpoint** | A specific URL or path where a service is reachable (e.g., `https://api.weather.com/v3/wx/forecast`) |

In [27]:
ask("ollama", "Explain APIs")

"**What is an API?**\n\nAn Application Programming Interface (API) is a set of defined rules that enables different software systems to communicate with each other. It allows one system to request services or data from another system, without sharing the underlying implementation details.\n\n**How does an API work?**\n\nHere's a step-by-step explanation of the API request-response cycle:\n\n1. **Request**: A client application (e.g., a web app, mobile app, or software) sends a request to the API, specifying the desired data or action.\n2. **API Gateway**: The request is routed to the API gateway, which acts as an entry point for the API.\n3. **API Processing**: The API gateway processes the request and determines the best course of action.\n4. **Data Retrieval**: If the request is for data retrieval, the API retrieves the required data from a database or other data storage system.\n5. **Response**: The API sends a response back to the client application, which may include the requested